In [ ]:
!nvidia-smi  # check GPU usage (e.g. is there already memory blocked)

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# this is important on our server, as the default /tmp directory is very small.
# so if you compile lots of models, it will eventually fill up.
# if you are working on your own machine, you might want/have to remove this!
if os.path.isdir("/var/tmp"):
    os.environ["TMPDIR"] = "/var/tmp"
    os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/var/tmp"

# Repository Showcase

Here, we show how to use the functionalities of our code repository to build datasets and models, as well as training the latter, without the CLI which we will otherwise be using in this course.

If you later want to adapt this to other models for debugging without the CLI (see "Lightning & CLI" in the `docs` folder for why you might want to do this), you can just import the corresponding `TrainingModule` and model instead, and change the initialization accordingly.
Of course you would need to remove the autoencoder-specific stuff (that looks at encoder/decoder etc.).
You can also try different datasets, of course;
in fact, some models may require specific transformations etc.

We have a video linked on E-Learning (or here: [LINK](https://cloud.ovgu.de/s/BaDbzLBQRgwE7yY)) that explores some of the options here!
To keep the notebook short, we do not include the entire thing in writing -- watch the video and play around with the code yourself.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

from lgm.data import ImageDataModule
from lgm.models.autoencoder import AETrainingModule, Autoencoder
from lgm.utils import interpolate
from lgm.visualization import ImagePlotter

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

## Data

Lightning uses "data modules" that wrap train/test datasets and loaders.
We of course recommend having a look at our implementation, but creating these modules is pretty simple.
We offer a bunch of datasets -- check the documentation for `ImageDataModule`.

In [ ]:
data_module = ImageDataModule("mnist", batch_size=32, num_workers=4)
data_module.prepare_data()
data_module.setup()

Dataloaders can be accessed via function calls (not attributes!):

In [ ]:
test_loader = data_module.test_dataloader()
inputs, labels = next(iter(test_loader))

print("input batch shape:", inputs.shape)

plotter = ImagePlotter(image_size=1.)
_ = plotter.image_grid(inputs,
                       title="Test set examples",
                       n_rows=4,
                       n_cols=8)

## Model

We have to differentiate between the `nn.Module` (i.e. base `torch` model) and the `LightningModule`.
In principle, the latter inherits from the former, so we could build all our models as "first class" `LightningModule` variations.
However, we prefer to have the _training_ separate from the actual _thing being trained_.
As such, we generally have a `LightningModule` that implements the training step, which has a `model` attribute that is just a regular `nn.Module`, and defines things like the `forward` function or the model architecture.

Our architectures will mostly inherit from certain base classes in the `lgm.layers.networks` module.
This contains classes like `EncoderNet`, `DecoderNet` or `EncoderDecoder`, which are appropriate foundations for most generative models.
Here, `Autoencoder` ist just an `EncoderDecoder` with a couple of extra steps.

There are various properties set in several `dataclass` objects.
Don't confuse these "configs" with the `yaml` files we use for the CLI!

In [ ]:
from lgm.models.autoencoder.likelihoods import GaussianFixedSigma
from lgm.layers import BlockConfig, BodyConfig, EncoderDecoderConfig


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device is", device)

likelihood = GaussianFixedSigma()

body_config = BodyConfig(blocks_per_level=2,
                         level_filters=[32, 64, 128])

block_config = BlockConfig()

config = EncoderDecoderConfig(input_shape=inputs.shape[1:],
                              latent_dim=16,
                              style="fc",
                              body_config=body_config,
                              block_config=block_config)

model = Autoencoder(likelihood,
                    enc_dec_config=config).to(device).eval()

We can plot the entire model architecture.
This is very long and can be hard to parse due to all the indentation, but it gives you an idea of the inner workings of the model.

In [ ]:
print(model)

We can also address various components, which are in turn `nn.Module` instances themselves.

In [ ]:
from lgm.utils import count_parameters

print(f"Model with {count_parameters(model):,d} parameters!")
print("...consisting of:")
print(f"Encoder with {count_parameters(model.encoder):,d} parameters!")
print(f"Decoder with {count_parameters(model.decoder):,d} parameters!")

Since we are dealing with an autoencoder, the output shape should be identical to the inputs, else something is wrong!

In [ ]:
out = model(inputs.to(device))
print("output shape", out.shape)

In [ ]:
encoder = model.encoder
decoder = model.decoder

The encoding shape is arbitrary and depends on various architectural choices.
We discuss this in the video.

In [ ]:
encoded = encoder(inputs.to(device))
print("encoded shape", encoded.shape)

Applying the decoder to the encoder outputs, however, should restore the input shape.

In [ ]:
decoded = decoder(encoded.to(device))
print("encoded shape", decoded.shape)

In [ ]:
encoder_cnn = encoder.cnn

In [ ]:
encoded_prehead = encoder_cnn(inputs.to(device))
print("encoded (before head) shape", encoded_prehead.shape)

## Training

For training, we require the core model built above, as well as an optimizer config;
then we can create the `LightningModule`.
Note that there are various ways to provide optimizers to Lightning, but this one has the best compatibility/flexibility with the CLI.

This entire thing is completely generic; as long as you use the correct model and lightning module, anything we train in this class is covered by the process below.

In [ ]:
from lgm.optimizers import OptimizationConfig, AdamWConfig, CosineWarmupConfig

optimization_config = OptimizationConfig(AdamWConfig(),
                                         CosineWarmupConfig())

In [ ]:
lightning_module = AETrainingModule(model, optimization_config)

Training is then simple.
We disable logging because this is just a showcase, plus there are some issues with converting certain dataclasses if we include a logger.
You will get some warning messages about no logger being configured, but that's okay.
The main question is whether training & validation run without crashing (and ideally with a decreasing loss).

In [ ]:
from lightning import Trainer
trainer = Trainer(max_epochs=2, logger=False)
trainer.fit(model=lightning_module, datamodule=data_module)